# 03 · Sector Screener

Run the full valuation pipeline across a universe of stocks and rank them  
by opportunity quality.

**Output:**
1. Opportunity map — Margin of Safety vs Confidence scatter  
2. Ranked bar chart — fair value vs current price  
3. Key-metrics heatmap — normalised fundamentals side by side  
4. Actionable shortlist — stocks in the buy zone  

Data is fetched in parallel. Each ticker takes ~5–10s on first run, then  
reads from disk cache. Full universe of 15 stocks: ~2–3 minutes cold, ~10s cached.

In [1]:
import sys
import pathlib

sys.path.insert(0, str(pathlib.Path().resolve().parent))

import warnings
warnings.filterwarnings("ignore")

from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from fairprice.data import FinancialsClient, MarketClient, MacroClient
from fairprice.nlp import SentimentClient
from fairprice.nlp.sentiment import active_backend
import fairprice.valuation as valuation_engine

# ── Universe definition ──────────────────────────────────────────────────────
UNIVERSE = {
    "Technology": ["AAPL", "MSFT", "GOOG", "META", "NVDA", "IBM", "AMD", "ORCL", "AMAT", "AVGO"],
    "Consumer":   ["AMZN", "WMT", "COST", "MCD", "NKE", "KO", "PEP", "VZ", "V"],
    "Healthcare": ["JNJ", "UNH", "PFE", "MDT"],
    "Financials": ["JPM", "BAC", "INTU"],
    "Energy":     ["XOM", "CVX"],
    "Defense":    ["GD", "GE", "RTX"],
}
ALL_TICKERS = [t for tickers in UNIVERSE.values() for t in tickers]
SECTOR_MAP  = {t: s for s, tickers in UNIVERSE.items() for t in tickers}

fin = FinancialsClient()
mkt = MarketClient()
mac = MacroClient()
nlp = SentimentClient()

macro = mac.get_macro_data()
print(f"Universe: {len(ALL_TICKERS)} tickers across {len(UNIVERSE)} sectors")
print(f"NLP backend: {active_backend()}")

Universe: 31 tickers across 6 sectors
NLP backend: keyword


In [2]:
def _value_one(ticker: str) -> dict | None:
    try:
        stmts = fin.get_statements(ticker)
        market = mkt.get_market_data(ticker)
        peers = fin.get_peers(ticker)
        _macro = mac.get_macro_data()  # cheap — from cache
        _macro.vix = market.vix

        result = valuation_engine.estimate(stmts, market, _macro, peers)

        ttm = stmts.ttm
        rev = ttm.get("Total Revenue", 0) or 0
        ni = ttm.get("Net Income", 0) or 0
        fcf = ttm.get("Free Cash Flow", 0) or ttm.get("Operating Cash Flow", 0) or 0

        return {
            "Ticker": ticker,
            "Sector": SECTOR_MAP.get(ticker, "Other"),
            "Price": round(market.current_price, 2),
            "Fair Value": result.fair_value_base,
            "FV Low": result.fair_value_low,
            "FV High": result.fair_value_high,
            "MoS %": round(result.margin_of_safety * 100, 1),
            "Confidence": round(result.confidence_score, 2),
            "Market Cap $B": round(market.market_cap / 1e9, 1),
            "Beta": round(market.beta_1y, 2),
            "FCF $B": round(fcf / 1e9, 1),
            "Net Margin %": round(ni / rev * 100, 1) if rev > 0 else None,
            "DCF": result.dcf_value,
            "Relative": result.relative_value,
            "ML": result.ml_value,
        }
    except Exception as exc:
        print(f"  {ticker}: failed — {exc}")
        return None


print("Fetching & valuing universe (parallel, ~2–3 min cold / 10s cached)...")

results = []
with ThreadPoolExecutor(max_workers=5) as pool:
    futures = {pool.submit(_value_one, t): t for t in ALL_TICKERS}
    done = 0
    for fut in as_completed(futures):
        done += 1
        r = fut.result()
        if r:
            results.append(r)
            print(
                f'  [{done:2d}/{len(ALL_TICKERS)}] {r["Ticker"]:6s}  '
                f'price=${r["Price"]:7.2f}  FV=${r["Fair Value"]:7.2f}  '
                f'MoS={r["MoS %"]:+5.1f}%  conf={r["Confidence"]:.2f}'
            )

df = pd.DataFrame(results).sort_values("MoS %", ascending=False)
print(f"\nScreened {len(df)} tickers successfully.")

Fetching & valuing universe (parallel, ~2–3 min cold / 10s cached)...
  [ 1/31] AAPL    price=$ 310.85  FV=$ 106.23  MoS=-192.6%  conf=0.61


FMP supplement failed for GOOG: 402 Client Error: Payment Required for url: https://financialmodelingprep.com/stable/income-statement?symbol=GOOG&limit=5&apikey=WYC2gJFtOZDy70xxCwtRSIUOK79YGa4J
FMP supplement failed for IBM: 402 Client Error: Payment Required for url: https://financialmodelingprep.com/stable/income-statement?symbol=IBM&limit=5&apikey=WYC2gJFtOZDy70xxCwtRSIUOK79YGa4J


  [ 2/31] GOOG    price=$ 384.83  FV=$ 147.89  MoS=-160.2%  conf=0.45
  [ 3/31] IBM     price=$ 255.20  FV=$ 347.91  MoS=+26.7%  conf=0.55


FMP supplement failed for ORCL: 402 Client Error: Payment Required for url: https://financialmodelingprep.com/stable/income-statement?symbol=ORCL&limit=5&apikey=WYC2gJFtOZDy70xxCwtRSIUOK79YGa4J


  [ 4/31] NVDA    price=$ 212.60  FV=$ 125.46  MoS=-69.5%  conf=0.23
  [ 5/31] ORCL    price=$ 190.96  FV=$ 222.72  MoS=+14.3%  conf=0.21
  [ 6/31] META    price=$ 635.25  FV=$ 719.34  MoS=+11.7%  conf=0.18
  [ 7/31] MSFT    price=$ 412.67  FV=$ 158.03  MoS=-161.1%  conf=0.48


...

  [22/31] UNH     price=$ 384.01  FV=$ 426.39  MoS= +9.9%  conf=0.48
  [23/31] INTU    price=$ 307.73  FV=$1628.40  MoS=+81.1%  conf=0.19
  [24/31] PFE     price=$  26.21  FV=$  38.78  MoS=+32.4%  conf=0.66


FMP supplement failed for GD: 402 Client Error: Payment Required for url: https://financialmodelingprep.com/stable/income-statement?symbol=GD&limit=5&apikey=WYC2gJFtOZDy70xxCwtRSIUOK79YGa4J


  [25/31] JPM     price=$ 299.28  FV=$ 273.50  MoS= -9.4%  conf=0.27
  [26/31] BAC     price=$  51.10  FV=$ 113.47  MoS=+55.0%  conf=0.65
  [27/31] GD      price=$ 342.69  FV=$ 616.25  MoS=+44.4%  conf=0.40


FMP supplement failed for RTX: 402 Client Error: Payment Required for url: https://financialmodelingprep.com/stable/income-statement?symbol=RTX&limit=5&apikey=WYC2gJFtOZDy70xxCwtRSIUOK79YGa4J


  [28/31] CVX     price=$ 182.40  FV=$ 137.04  MoS=-33.1%  conf=0.64
  [29/31] RTX     price=$ 176.59  FV=$ 352.22  MoS=+49.9%  conf=0.15
  [30/31] XOM     price=$ 147.90  FV=$ 137.34  MoS= -7.7%  conf=0.66
  [31/31] GE      price=$ 317.21  FV=$ 126.64  MoS=-150.5%  conf=0.52

Screened 31 tickers successfully.


In [3]:
df

,Ticker,Sector,Price,Fair Value,FV Low,FV High,MoS %,Confidence,Market Cap $B,Beta,FCF $B,Net Margin %,DCF,Relative,ML
15,VZ,Consumer,48.24,1068.21,331.44,1999.19,95.5,0.16,201.4,-0.14,19.9,12.5,1601.06,None,215.65
22,INTU,Financials,307.73,1628.40,828.51,3209.97,81.1,0.19,84.2,0.49,7.7,21.9,2403.37,None,388.45
16,PEP,Consumer,147.74,392.54,190.56,769.65,62.4,0.18,202.0,-0.06,8.8,9.1,582.36,None,88.82
25,BAC,Financials,51.10,113.47,87.76,142.58,55.0,0.65,362.6,0.93,56.6,27.3,116.56,None,108.52
19,MDT,Healthcare,75.98,156.55,111.89,204.22,51.5,0.67,97.5,0.34,5.4,13.0,155.48,None,158.28
12,MCD,Consumer,280.92,574.37,325.49,979.74,51.1,0.50,199.6,0.09,7.0,31.6,683.97,None,399.00
28,RTX,Defense,176.59,352.22,165.16,734.90,49.9,0.15,237.8,0.50,8.0,8.0,531.10,None,66.01
26,GD,Defense,342.69,616.25,431.72,827.67,44.4,0.40,92.7,0.53,6.2,8.1,791.66,None,335.59
23,PFE,Healthcare,26.21,38.78,29.48,50.99,32.4,0.66,149.4,0.42,9.5,11.8,37.95,None,40.12
20,JNJ,Healthcare,231.29,337.59,216.80,498.70,31.5,0.48,556.8,0.07,17.4,21.8,406.62,None,227.14


## 1 · Opportunity Map

The best opportunities sit in the **top-right quadrant**: high margin of safety  
(stock is cheap relative to fair value) *and* high confidence (data is complete  
and models agree).

- **Bubble size** = market cap  
- **Colour** = sector  
- **Dashed lines** = threshold filters (MoS > 15%, confidence > 0.5)

In [4]:
fig = px.scatter(
    df,
    x="Confidence",
    y="MoS %",
    color="Sector",
    size="Market Cap $B",
    size_max=30,
    text="Ticker",
    hover_data=["Price", "Fair Value", "FCF $B", "Net Margin %"],
    title="Opportunity Map — Margin of Safety vs Confidence  (size = market cap)",
    color_discrete_sequence=px.colors.qualitative.Bold,
)
fig.add_hline(y=15,  line_dash="dash", line_color="#4CAF50", line_width=1.5,
              annotation_text="MoS +15%", annotation_position="right")
fig.add_hline(y=-20, line_dash="dash", line_color="#F44336", line_width=1.5,
              annotation_text="MoS −20%", annotation_position="right")
fig.add_vline(x=0.5, line_dash="dash", line_color="#9E9E9E", line_width=1.5,
              annotation_text="Conf 0.5")
fig.update_traces(textposition="top center")
fig.update_layout(
    height=580,
    xaxis_title="Confidence Score  →",
    yaxis_title="Margin of Safety %",
)
fig.show()

## 2 · Ranked Fair Value vs Current Price

In [5]:
df_sorted = df.sort_values("MoS %", ascending=True).reset_index(drop=True)

fig = go.Figure()

bar_colors = ["#4CAF50" if m > 0 else "#F44336" for m in df_sorted["MoS %"]]

fig.add_trace(
    go.Bar(
        name="Fair Value (base)",
        y=df_sorted["Ticker"],
        x=df_sorted["Fair Value"],
        orientation="h",
        marker_color=bar_colors,
        opacity=0.7,
        text=[f"${v:.0f}" for v in df_sorted["Fair Value"]],
        textposition="outside",
    )
)

# Fair value range bars
for _, row in df_sorted.iterrows():
    fig.add_shape(
        type="line",
        x0=row["FV Low"],
        x1=row["FV High"],
        y0=row["Ticker"],
        y1=row["Ticker"],
        line=dict(color="rgba(0,0,0,0.3)", width=3),
    )

# Current price dots
fig.add_trace(
    go.Scatter(
        name="Current Price",
        y=df_sorted["Ticker"],
        x=df_sorted["Price"],
        mode="markers",
        marker=dict(color="#212121", size=10, symbol="diamond"),
    )
)

fig.update_layout(
    title="Fair Value Range vs Current Price  (bars = base; lines = low–high range; ◆ = market price)",
    xaxis_title="Per-Share Value ($)",
    height=max(400, len(df_sorted) * 38),
    barmode="overlay",
    legend=dict(orientation="h", y=1.05),
)
fig.show()

## 3 · Fundamentals Heatmap

Normalised key metrics across the universe.  
Helps spot outliers — e.g. unusually high beta or thin margins —  
that might explain why a model gives a low confidence score.

In [6]:
metrics = ["MoS %", "Confidence", "FCF $B", "Net Margin %", "Beta", "Market Cap $B"]
df_heat = df.set_index("Ticker")[metrics].copy()

# Z-score normalise each column so colour scale is comparable
df_norm = df_heat.apply(
    lambda col: (col - col.mean()) / col.std() if col.std() > 0 else col
)

# Invert Beta (lower beta = better, relatively) for visual consistency
if "Beta" in df_norm.columns:
    df_norm["Beta"] = -df_norm["Beta"]

annotations = [
    [
        f"{df_heat.loc[t, m]:.1f}" if pd.notna(df_heat.loc[t, m]) else "N/A"
        for m in metrics
    ]
    for t in df_norm.index
]

fig = go.Figure(
    go.Heatmap(
        z=df_norm.values,
        x=metrics,
        y=df_norm.index.tolist(),
        text=annotations,
        texttemplate="%{text}",
        textfont=dict(size=10),
        colorscale="RdYlGn",
        zmid=0,
        colorbar=dict(title="Z-score"),
    )
)
fig.update_layout(
    title="Normalised Fundamentals Heatmap  (green = relatively strong)",
    height=max(350, len(df_norm) * 30 + 80),
    xaxis_tickangle=-25,
)
fig.show()

## 4 · Actionable Shortlist

Stocks that pass **all three filters** simultaneously:
- Margin of Safety > 15% (stock trades below estimated fair value)
- Confidence score ≥ 0.50 (models have adequate data and agree reasonably)
- Bear scenario fair value still above current price (downside cushion)

In [7]:
# ── Filter thresholds ─────────────────────────────────────────────────────────
MIN_MOS = 15.0  # % — minimum margin of safety
MIN_CONF = 0.50  # confidence score
BEAR_FLOOR = True  # require bear scenario > current price

buy_zone = df[
    (df["MoS %"] > MIN_MOS)
    & (df["Confidence"] >= MIN_CONF)
    & (df["FV Low"] > df["Price"] if BEAR_FLOOR else True)
].sort_values("MoS %", ascending=False)

avoid_zone = df[(df["MoS %"] < -20) & (df["Confidence"] >= 0.45)].sort_values("MoS %")

display_cols = [
    "Ticker",
    "Sector",
    "Price",
    "Fair Value",
    "FV Low",
    "FV High",
    "MoS %",
    "Confidence",
    "FCF $B",
    "Net Margin %",
]

print("=" * 65)
print("  BUY ZONE  (MoS > 15%, confidence ≥ 0.5, bear FV > price)")
print("=" * 65)
if buy_zone.empty:
    print("  No stocks meet all three criteria in this universe.")
    print("  Try loosening thresholds or expanding the universe.")
else:
    print(buy_zone[display_cols].to_string(index=False))

print()
print("─" * 65)
print("  AVOID / OVERVALUED  (MoS < −20%, confidence ≥ 0.45)")
print("─" * 65)
if avoid_zone.empty:
    print("  None identified at current prices.")
else:
    print(avoid_zone[display_cols].to_string(index=False))

print()
print("─" * 65)
print("  FULL UNIVERSE SUMMARY")
print("─" * 65)
summary = df[
    ["Ticker", "Sector", "Price", "Fair Value", "MoS %", "Confidence"]
].to_string(index=False)
print(summary)

  BUY ZONE  (MoS > 15%, confidence ≥ 0.5, bear FV > price)
Ticker     Sector  Price  Fair Value  FV Low  FV High  MoS %  Confidence  FCF $B  Net Margin %
   BAC Financials  51.10      113.47   87.76   142.58   55.0        0.65    56.6          27.3
   MDT Healthcare  75.98      156.55  111.89   204.22   51.5        0.67     5.4          13.0
   MCD   Consumer 280.92      574.37  325.49   979.74   51.1        0.50     7.0          31.6
   PFE Healthcare  26.21       38.78   29.48    50.99   32.4        0.66     9.5          11.8

─────────────────────────────────────────────────────────────────
  AVOID / OVERVALUED  (MoS < −20%, confidence ≥ 0.45)
─────────────────────────────────────────────────────────────────
Ticker     Sector  Price  Fair Value  FV Low  FV High  MoS %  Confidence  FCF $B  Net Margin %
  AMAT Technology 448.25       70.28   57.89    84.72 -537.8        0.62     5.3          29.3
   NKE   Consumer  45.98        8.61    7.71     9.85 -434.1        0.57     1.0         

In [8]:
# Save results to CSV for further analysis
out_path = pathlib.Path().resolve() / "screener_results.csv"
df.to_csv(out_path, index=False)
print(f"Results saved to: {out_path}")

# Quick stats
print(f"\nUniverse stats:")
print(f'  Median MoS          : {df["MoS %"].median():+.1f}%')
print(f'  Median confidence   : {df["Confidence"].median():.2f}')
print(f'  Undervalued (>0% MoS): {(df["MoS %"] > 0).sum()} / {len(df)}')
print(f"  In buy zone         : {len(buy_zone)}")
print(f"  In avoid zone       : {len(avoid_zone)}")

Results saved to: /Users/aleksandra/Documents/playground/Code/FairPrice/notebooks/screener_results.csv

Universe stats:
  Median MoS          : -7.6%
  Median confidence   : 0.45
  Undervalued (>0% MoS): 15 / 31
  In buy zone         : 4
  In avoid zone       : 10


## 5 · Sentiment Signal — Shortlist Deep-Dive

For the stocks in the **buy zone** and **avoid zone** we fetch live sentiment and
compute a **composite signal**:

```
composite = 0.60 × tanh(MoS% / 50)  +  0.40 × sentiment_score
```

- `tanh` compresses extreme MoS values so a +200 % outlier doesn't dominate
- 60 % weight on fundamentals, 40 % on news momentum

> **Actionable rule**: composite > 0.3 with confidence ≥ 0.5 → high-conviction long.
> Composite < −0.3 with confidence ≥ 0.5 → strong avoid.

In [9]:
focus = pd.concat([buy_zone, avoid_zone], ignore_index=True)
print(f"Fetching sentiment for {len(focus)} tickers  [{active_backend()} backend]...\n")

def _get_sent(ticker):
    s = nlp.get_sentiment(ticker)
    return {
        "Ticker":      ticker,
        "Sent Score":  round(s.sentiment_score, 3),
        "Pos %":       round(s.positive_ratio * 100, 1),
        "Neg %":       round(s.negative_ratio * 100, 1),
        "Uncertainty": round(s.uncertainty_score, 2),
        "Guidance":    s.guidance_revision or "—",
        "N Articles":  s.n_articles + s.n_alpaca_articles,
    }

with ThreadPoolExecutor(max_workers=5) as pool:
    sent_rows = list(pool.map(_get_sent, focus["Ticker"].tolist()))

df_sent_focus = pd.DataFrame(sent_rows)
df_merged     = focus.merge(df_sent_focus, on="Ticker")

df_merged["MoS norm"]  = np.tanh(df_merged["MoS %"] / 50)
df_merged["Composite"] = (0.60 * df_merged["MoS norm"] + 0.40 * df_merged["Sent Score"]).round(3)

for _, row in df_merged.sort_values("Composite", ascending=False).iterrows():
    zone = "BUY " if row["Ticker"] in buy_zone["Ticker"].values else "AVOID"
    print(
        f"  [{zone}] {row['Ticker']:6s}  MoS={row['MoS %']:+5.1f}%  "
        f"sent={row['Sent Score']:+.3f}  unc={row['Uncertainty']:.2f}  "
        f"guidance={row['Guidance']:<12}  composite={row['Composite']:.3f}"
    )

Fetching sentiment for 14 tickers  [keyword backend]...

  [BUY ] MCD     MoS=+51.1%  sent=+0.257  unc=0.00  guidance=—             composite=0.565
  [BUY ] BAC     MoS=+55.0%  sent=+0.100  unc=0.07  guidance=—             composite=0.520
  [BUY ] MDT     MoS=+51.5%  sent=+0.043  unc=0.00  guidance=—             composite=0.482
  [BUY ] PFE     MoS=+32.4%  sent=+0.090  unc=0.03  guidance=—             composite=0.378
  [AVOID] KO      MoS=-31.2%  sent=+0.043  unc=0.02  guidance=—             composite=-0.315
  [AVOID] CVX     MoS=-33.1%  sent=+0.040  unc=0.01  guidance=—             composite=-0.332
  [AVOID] AVGO    MoS=-407.8%  sent=+0.210  unc=0.05  guidance=—             composite=-0.516
  [AVOID] AMAT    MoS=-537.8%  sent=+0.180  unc=0.09  guidance=—             composite=-0.528
  [AVOID] AAPL    MoS=-192.6%  sent=+0.156  unc=0.04  guidance=reiterated    composite=-0.537
  [AVOID] WMT     MoS=-156.5%  sent=+0.140  unc=0.07  guidance=lowered       composite=-0.542
  [AVOID] NKE    

In [10]:
# ── Quadrant scatter: Valuation × Sentiment ──────────────────────────────────
bz_tickers = set(buy_zone["Ticker"])
az_tickers = set(avoid_zone["Ticker"])

fig = go.Figure()
for _, row in df_merged.iterrows():
    t     = row["Ticker"]
    color = "#4CAF50" if t in bz_tickers else "#F44336"
    zone  = "Buy zone" if t in bz_tickers else "Avoid zone"
    fig.add_trace(go.Scatter(
        x=[row["Sent Score"]], y=[row["MoS %"]],
        mode="markers+text", text=[t], textposition="top center",
        marker=dict(
            size=12 + row["Confidence"] * 24, color=color, opacity=0.82,
            line=dict(color="white", width=1.5),
        ),
        name=zone, legendgroup=zone,
        showlegend=t == df_merged.iloc[0]["Ticker"],
    ))

fig.add_hline(y=15,  line_dash="dash", line_color="#4CAF50",
              annotation_text="MoS +15%", annotation_position="right")
fig.add_hline(y=-20, line_dash="dash", line_color="#F44336",
              annotation_text="MoS −20%", annotation_position="right")
fig.add_vline(x=0, line_dash="dash", line_color="#9E9E9E")

for ax, ay, txt, bg in [
    ( 0.40,  60, "★ BUY ZONE<br>cheap + bullish",       "rgba(76,175,80,0.10)"),
    (-0.40,  60, "⚡ VALUE TRAP?<br>cheap + bearish",    "rgba(255,152,0,0.10)"),
    ( 0.40, -60, "⚠ MOMENTUM TRAP<br>expensive+bullish","rgba(255,87,34,0.10)"),
    (-0.40, -60, "▼ CLEAR AVOID<br>expensive+bearish",  "rgba(244,67,54,0.10)"),
]:
    fig.add_annotation(x=ax, y=ay, text=txt, showarrow=False,
                       font=dict(size=10), bgcolor=bg, borderpad=3)

fig.update_layout(
    title="Valuation × Sentiment Quadrant  (bubble size = confidence score)",
    xaxis_title="Sentiment Score  (← bearish … bullish →)",
    yaxis_title="Margin of Safety %",
    height=520,
)
fig.show()

# ── Composite ranking table ───────────────────────────────────────────────────
cols = ["Ticker", "Sector", "Price", "MoS %", "Confidence",
        "Sent Score", "Uncertainty", "Guidance", "Composite"]
df_final = df_merged[cols].sort_values("Composite", ascending=False).reset_index(drop=True)
print("\nComposite signal ranking:")
print(df_final.to_string(index=False))

top    = df_final[(df_final["Composite"] > 0.3)  & (df_final["Confidence"] >= 0.45)]
bottom = df_final[(df_final["Composite"] < -0.3) & (df_final["Confidence"] >= 0.45)]
if not top.empty:
    print(f"\n★  High-conviction longs : {', '.join(top['Ticker'].tolist())}")
if not bottom.empty:
    print(f"▼  High-conviction avoids: {', '.join(bottom['Ticker'].tolist())}")


Composite signal ranking:
Ticker     Sector  Price  MoS %  Confidence  Sent Score  Uncertainty   Guidance  Composite
   MCD   Consumer 280.92   51.1        0.50       0.257         0.00          —      0.565
   BAC Financials  51.10   55.0        0.65       0.100         0.07          —      0.520
   MDT Healthcare  75.98   51.5        0.67       0.043         0.00          —      0.482
   PFE Healthcare  26.21   32.4        0.66       0.090         0.03          —      0.378
    KO   Consumer  81.62  -31.2        0.50       0.043         0.02          —     -0.315
   CVX     Energy 182.40  -33.1        0.64       0.040         0.01          —     -0.332
  AVGO Technology 421.86 -407.8        0.45       0.210         0.05          —     -0.516
  AMAT Technology 448.25 -537.8        0.62       0.180         0.09          —     -0.528
  AAPL Technology 310.85 -192.6        0.61       0.156         0.04 reiterated     -0.537
   WMT   Consumer 118.54 -156.5        0.45       0.140        